<a href="https://www.kaggle.com/code/gberi22/notebookdf63c57883?scriptVersionId=325493843" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/gberi22/speech-and-language-processing/book.txt


config.py

In [2]:
%%writefile config.py
import os
import torch


def _int(name, default):
    return int(os.environ.get(name, default))


def _flt(name, default):
    return float(os.environ.get(name, default))


SEED = _int("NS_SEED", 42)

DATA_DIR = os.environ.get("NS_DATA_DIR", "data")
BOOK_PATH = os.path.join(DATA_DIR, "book.txt")
CHUNKS_PATH = os.path.join(DATA_DIR, "chunks.json")
CORPUS_PATH = os.path.join(DATA_DIR, "book.txt")
TRAIN_PAIRS_PATH = os.path.join(DATA_DIR, "train_pairs.json")
VAL_EVAL_PATH = os.path.join(DATA_DIR, "val_eval.json")
TEST_EVAL_PATH = os.path.join(DATA_DIR, "test_eval.json")
TOKENIZER_PATH = os.environ.get("NS_TOKENIZER", "tokenizer.json")
MODEL_PATH = os.environ.get("NS_MODEL", "best_model.pt")

CHUNK_MIN = _int("NS_CHUNK_MIN", 200)
CHUNK_MAX = _int("NS_CHUNK_MAX", 300)

VOCAB_SIZE = _int("NS_VOCAB_SIZE", 8000)

D_MODEL = _int("NS_D_MODEL", 256)
NHEAD = _int("NS_NHEAD", 4)
NUM_LAYERS = _int("NS_NUM_LAYERS", 4)
DIM_FF = _int("NS_DIM_FF", 512)
DROPOUT = _flt("NS_DROPOUT", 0.1)
MAX_LEN = _int("NS_MAX_LEN", 320)
MODEL_MAX_LEN = _int("NS_MODEL_MAX_LEN", 512)

PRETRAIN_EPOCHS = _int("NS_PRETRAIN_EPOCHS", 3)
PRETRAIN_BS = _int("NS_PRETRAIN_BS", 64)
PRETRAIN_LR = _flt("NS_PRETRAIN_LR", 1e-4)
FINETUNE_EPOCHS = _int("NS_FINETUNE_EPOCHS", 5)
FINETUNE_BS = _int("NS_FINETUNE_BS", 32)
FINETUNE_LR = _flt("NS_FINETUNE_LR", 1e-4)
TEMPERATURE = _flt("NS_TEMPERATURE", 0.05)

EVAL_K = _int("NS_EVAL_K", 10)
PAIRS_PER_CHUNK = _int("NS_PAIRS_PER_CHUNK", 3)

DEVICE = os.environ.get("NS_DEVICE", "cuda" if torch.cuda.is_available() else "cpu")


Writing config.py


In [3]:
import config, os, shutil, glob

os.makedirs("data", exist_ok=True)
src = glob.glob('/kaggle/input/**/book.txt', recursive=True)[0]
shutil.copy(src, config.CORPUS_PATH)
print("copied:", src, "->", os.listdir("data"))

copied: /kaggle/input/datasets/gberi22/speech-and-language-processing/book.txt -> ['book.txt']


model.py

In [4]:
%%writefile model.py
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


def masked_mean(hidden, attention_mask):
    mask = attention_mask.unsqueeze(-1).float()         
    summed = (hidden * mask).sum(dim=1)                   
    counts = mask.sum(dim=1).clamp(min=1e-9)              
    return summed / counts
 
 
class PositionalEncoding(nn.Module): 
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))       
 
    def forward(self, x):                                 
        return x + self.pe[:, : x.size(1)]


class MultiHeadSelfAttention(nn.Module): 
    def __init__(self, d_model, nhead, dropout=0.1):
        super().__init__()
        assert d_model % nhead == 0, "d_model must be divisible by nhead"
        self.nhead = nhead
        self.d_head = d_model // nhead
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
 
    def forward(self, x, key_padding_mask):
        B, L, d = x.shape
        def split(t):
            return t.view(B, L, self.nhead, self.d_head).transpose(1, 2)
        q, k, v = split(self.q_proj(x)), split(self.k_proj(x)), split(self.v_proj(x))
 
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_head)
        if key_padding_mask is not None:
            mask = key_padding_mask.view(B, 1, 1, L)                  
            scores = scores.masked_fill(mask, float("-inf"))
        attention = torch.softmax(scores, dim=-1)
        attention = self.dropout(attention)
        ctx = attention @ v                                                
        ctx = ctx.transpose(1, 2).contiguous().view(B, L, d)          
        return self.out_proj(ctx)

class TransformerBlock(nn.Module): 
    def __init__(self, d_model, nhead, dim_feedforward, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadSelfAttention(d_model, nhead, dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, dim_feedforward),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim_feedforward, d_model),
        )
        self.dropout = nn.Dropout(dropout)
 
    def forward(self, x, key_padding_mask):
        x = x + self.dropout(self.attn(self.norm1(x), key_padding_mask))
        x = x + self.dropout(self.ff(self.norm2(x)))
        return x
 
 
class TransformerEncoderModel(nn.Module):
    def __init__(self, vocab_size, d_model=256, nhead=4, num_layers=4,
                 dim_feedforward=512, max_len=512, dropout=0.1, pad_id=0):
        super().__init__()
        self.pad_id = pad_id
        self.d_model = d_model
        self.embed = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.pos = PositionalEncoding(d_model, max_len)
        self.in_dropout = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, nhead, dim_feedforward, dropout)
            for _ in range(num_layers)
        ])
        self.final_norm = nn.LayerNorm(d_model)
 
    def forward(self, input_ids, attention_mask):
        x = self.embed(input_ids) * math.sqrt(self.d_model)
        x = self.in_dropout(self.pos(x))
        key_padding_mask = attention_mask == 0              
        for block in self.blocks:
            x = block(x, key_padding_mask)
        x = self.final_norm(x)
        return masked_mean(x, attention_mask)         

class BagOfEmbeddings(nn.Module):
    def __init__(self, vocab_size, d_model=256, pad_id=0, dropout=0.1):
        super().__init__()
        self.pad_id = pad_id
        self.d_model = d_model
        self.embed = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.dropout = nn.Dropout(dropout)
 
    def forward(self, input_ids, attention_mask):
        x = self.dropout(self.embed(input_ids))
        return masked_mean(x, attention_mask)

Writing model.py


contrastive_learning.py

In [5]:
%%writefile contrastive_learning.py

import torch
import torch.nn.functional as F
 
 
def info_nce(query_emb, passage_emb, temperature=0.05):
    q = F.normalize(query_emb, dim=-1)
    p = F.normalize(passage_emb, dim=-1)
    logits = (q @ p.t()) / temperature            
    labels = torch.arange(q.size(0), device=q.device)
    loss_q = F.cross_entropy(logits, labels)      
    loss_p = F.cross_entropy(logits.t(), labels)  
    return 0.5 * (loss_q + loss_p)
    

Writing contrastive_learning.py


**build_tokenizer.py** - train BPE tokenizer + load/encode helpers, whole point of this file is to handle rare/unseen words and build an vocabulary

In [6]:
%%writefile build_tokenizer.py

from typing import Callable, List, Tuple

from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

import config


SPECIAL_TOKENS = ["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"]

def train_tokenizer(files: List[str], vocab_size: int = config.VOCAB_SIZE, out_path: str = config.TOKENIZER_PATH) -> Tokenizer:
    tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))
    tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()
    tokenizer.decoder = decoders.BPEDecoder()
    trainer = trainers.BpeTrainer(vocab_size=vocab_size, special_tokens=SPECIAL_TOKENS)
    tokenizer.train(files, trainer)
    tokenizer.save(out_path)
    print(f"saved tokenizer -> {out_path}  (vocab={tokenizer.get_vocab_size()})")
    return tokenizer

def load_tokenizer(path: str = config.TOKENIZER_PATH) -> Tuple[Callable[[str], List[int]], int, int]:
    tokenizer = Tokenizer.from_file(path)
    pad_id = tokenizer.token_to_id("[PAD]")
    vocab_size = tokenizer.get_vocab_size()
    tokenize = lambda text: tokenizer.encode(text).ids
    return tokenize, pad_id, vocab_size

def main() -> None:
    train_tokenizer([config.CORPUS_PATH])
    

Writing build_tokenizer.py


**data.py** - turns list of token-ID sequennces into padded pytorch batches encoder can consume. Its gonna be a bridge between the tokenizer and the model

In [7]:
%%writefile data.py

from typing import Callable, List, Tuple

import torch
from torch import Tensor
from torch.utils.data import Dataset


class PairDataset(Dataset):

    def __init__(self, pairs, tokenize: Callable[[str], List[int]], max_length: int = 128) -> None:
        self.pairs = pairs
        self.tokenize = tokenize
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.pairs)

    def __getitem__(self, index: int) -> Tuple[List[int], List[int]]:
        query, passage = self.pairs[index]
        return self.tokenize(query)[: self.max_length], self.tokenize(passage)[: self.max_length]


class TextDataset(Dataset):

    def __init__(self, texts, tokenize: Callable[[str], List[int]], max_length: int = 128) -> None:
        self.texts = texts
        self.tokenize = tokenize
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, index: int) -> List[int]:
        return self.tokenize(self.texts[index])[: self.max_length]


def pad_batch(sequences: List[List[int]], pad_id: int = 0) -> Tuple[Tensor, Tensor]:
    sequences = [seq if len(seq) > 0 else [pad_id] for seq in sequences]
    max_length = max(len(seq) for seq in sequences)
    input_ids = torch.full((len(sequences), max_length), pad_id, dtype=torch.long)
    attention_mask = torch.zeros((len(sequences), max_length), dtype=torch.long)
    for index, seq in enumerate(sequences):
        input_ids[index, : len(seq)] = torch.tensor(seq, dtype=torch.long)
        attention_mask[index, : len(seq)] = 1
    return input_ids, attention_mask

def pair_collate(batch, pad_id: int = 0) -> Tuple[Tuple[Tensor, Tensor], Tuple[Tensor, Tensor]]:
    queries, passages = zip(*batch)
    return pad_batch(queries, pad_id), pad_batch(passages, pad_id)

def text_collate(batch, pad_id: int = 0) -> Tuple[Tensor, Tensor]:
    return pad_batch(batch, pad_id)

Writing data.py


**chunk_book.py** - This file cuts the book into 200–300-word chunks that become the actual units which search engine retrieves and ranks

In [8]:
%%writefile chunk_book.py

import json
import os
import re
from typing import List

import config


def split_sentences(text: str) -> List[str]:
    text = re.sub(r"\s+", " ", text).strip()
    parts = re.split(r"(?<=[.!?])\s+", text)
    return [part.strip() for part in parts if part.strip()]

def chunk_text(text: str, min_words: int = config.CHUNK_MIN, max_words: int = config.CHUNK_MAX) -> List[str]:
    chunks, current, current_words = [], [], 0
    for sentence in split_sentences(text):
        word_count = len(sentence.split())
        if current_words + word_count > max_words and current_words >= min_words:
            chunks.append(" ".join(current))
            current, current_words = [], 0
        current.append(sentence)
        current_words += word_count
    if current and current_words >= max(1, min_words // 4):
        chunks.append(" ".join(current))
    return chunks

def main() -> None:
    os.makedirs(config.DATA_DIR, exist_ok=True)
    with open(config.BOOK_PATH, encoding="utf-8") as book_file:
        text = book_file.read()
    chunks = chunk_text(text)
    with open(config.CHUNKS_PATH, "w", encoding="utf-8") as chunks_file:
        json.dump(chunks, chunks_file, ensure_ascii=False)
    with open(config.CORPUS_PATH, "w", encoding="utf-8") as corpus_file:
        corpus_file.write("\n".join(chunks))
    print(f"chunked into {len(chunks)} passages -> {config.CHUNKS_PATH}")


Writing chunk_book.py


**build_pairs.py** - Turn the book's chunks into training pairs + eval sets. This file is the bridge between raw chunks and what the model actually trains and is scored on

In [9]:
%%writefile build_pairs.py

import json
import random
from typing import Dict, List

import config
from chunk_book import split_sentences


def ict_pairs(chunk: str, n_per_chunk: int) -> List[List[str]]:
    sentences = split_sentences(chunk)
    if len(sentences) < 2:
        return []
    pairs = []
    picked_indices = random.sample(range(len(sentences)), k=min(n_per_chunk, len(sentences)))
    for index in picked_indices:
        query = sentences[index]
        positive = " ".join(sentences[:index] + sentences[index + 1:])
        if positive.strip():
            pairs.append([query, positive])
    return pairs

def eval_queries(chunk: str, num_queries: int = 1) -> List[str]:
    sentences = split_sentences(chunk)
    if not sentences:
        return []
    return random.sample(sentences, k=min(num_queries, len(sentences)))

def main() -> None:
    random.seed(config.SEED)
    with open(config.CHUNKS_PATH, encoding="utf-8") as chunks_file:
        chunks = json.load(chunks_file)

    indices = list(range(len(chunks)))
    random.shuffle(indices)
    num_chunks = len(indices)
    num_train = int(0.8 * num_chunks)
    num_val = int(0.1 * num_chunks)
    train_indices = indices[:num_train]
    val_indices = indices[num_train:num_train + num_val]
    test_indices = indices[num_train + num_val:]

    train_pairs = []
    for index in train_indices:
        train_pairs.extend(ict_pairs(chunks[index], config.PAIRS_PER_CHUNK))

    def build_eval(split_indices: List[int]) -> Dict[str, list]:
        queries, gold = [], []
        for index in split_indices:
            for query in eval_queries(chunks[index], num_queries=1):
                queries.append(query)
                gold.append([index]) 
        return {"queries": queries, "gold": gold}

    with open(config.TRAIN_PAIRS_PATH, "w", encoding="utf-8") as train_file:
        json.dump(train_pairs, train_file, ensure_ascii=False)
    with open(config.VAL_EVAL_PATH, "w", encoding="utf-8") as val_file:
        json.dump(build_eval(val_indices), val_file, ensure_ascii=False)
    with open(config.TEST_EVAL_PATH, "w", encoding="utf-8") as test_file:
        json.dump(build_eval(test_indices), test_file, ensure_ascii=False)

    print(f"train pairs: {len(train_pairs)} | val chunks: {len(val_indices)} | "
          f"test chunks: {len(test_indices)}")


Writing build_pairs.py


**prepare_data.py** - This file runs whole data pipeline in order.

In [10]:
import chunk_book
import build_tokenizer
import build_pairs


def main() -> None:
    print("1/3 chunking book ...")
    chunk_book.main()
    print("2/3 training tokenizer ...")
    build_tokenizer.main()
    print("3/3 building pairs + eval sets ...")
    build_pairs.main()
    print("data prep complete.")

if __name__ == "__main__":
    main()

1/3 chunking book ...
chunked into 1095 passages -> data/chunks.json
2/3 training tokenizer ...



saved tokenizer -> tokenizer.json  (vocab=8000)
3/3 building pairs + eval sets ...
train pairs: 2614 | val chunks: 109 | test chunks: 110
data prep complete.
